# 04 - Entity matching and data filtering

This notebook has two main goals:

1. Integrate metadata for each intervention and save an unfiltered dataset.
2. Filter the dataset to meet the needs of this research.

> **INPUT:**  
- `data/raw/interventions/PL_<leg>_<num>.txt` — files, each containing every intervention (and its author's name)  
- `data/external/diputados.json` — JSON file retrieved from the Congress website containing deputy data
- `data/external/dates.csv` — list containing the date of each session (obtained in *01_scraping_downloads*)

> **OUTPUT:**  
- `data/processed/interventions_raw.csv` — all interventions with full metadata (legislature, session number, deputy, group, gender, date)  
- `data/processed/interventions_filtered.csv` — filtered interventions with normalized metadata (according to the requirements of this research)

**IMPORTANT**: Both raw and unfiltered datasets are complete in the repo, availaible for download.

In [5]:
from pathlib import Path
import json

# Change the input paths HERE:
INTERVENTION_TEXT_DIR = Path('..') / 'data' / 'raw' / 'interventions'  
DATES_FILE = Path('..') / 'data' / 'external' / 'dates.csv'
DEPUTIES_JSON = Path('..') / 'data' / 'external' / 'diputados.json'

# Change the output path HERE:
OUTPUT_DIR = Path('..') / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Loading data

import pandas as pd

with open(DEPUTIES_JSON, 'r', encoding='utf-8') as f:
    deputies_data = json.load(f)

diputados = deputies_data.get('diputados', [])
cargos = deputies_data.get('cargos', [])  # Members with an organizational role (president, secretaries, ...)

print(f'Loaded {len(diputados)} deputies from {DEPUTIES_JSON.name}')
print(f'Loaded {len(cargos)} organizational positions','\n\n','----------------------------------','\n')

dates_df = pd.read_csv(DATES_FILE)
print(f'Loaded dates for {len(dates_df)} sessions\n')
dates_df.head()


### Entity matching

We need to match every name of our txt files to a deputy from the JSON file.

The main issue is that due to the nature of the files, most names aren't correct character by character.
The chosen solution is to use the method `SequenceMatcher.ratio()` from the *difflib* module, which gives a similarity ratio (0-1) between two strings.

**Versions**: `difflib from python: 3.12.2`

In [ ]:
import re
from difflib import SequenceMatcher
from typing import Dict, Any, List, Tuple

def best_match(tag: str, candidates: List[Dict]) -> Dict[str, Any]:
    if not tag or not tag.strip():
        return {'name': None, 'group': None, 'gender': None, 'similarity': -1, 'diputado': {}}
    
    # Localizar marcadores en el tag
    paren_l = tag.find('(')
    paren_r = tag.find(')')
    colon_pos = tag.find(':')
    
    # Si no hay dos puntos, asumir que continúa hasta el final
    if colon_pos < 0:
        colon_pos = len(tag)
    
    best_sim = -1
    best_match_name = None
    best_grupo = None
    best_gender = None
    best_diputado = {}
    
    for candidate in candidates:
        apellidos = candidate.get('apellidos', '').lower().strip()
        if not apellidos:
            continue
        
        # Probar varias extracciones:
        # 1. Entre paréntesis (si existen)
        # 2. Después de 'El señor' o 'La señora' (9+) hasta ':' o final
        # 3. Todo el tag
        
        tag_lower = tag.lower()
        extractions = []
        
        if paren_l >= 0 and paren_r > paren_l:
            extractions.append(tag[paren_l+1:paren_r].lower().strip())
        
        # Erase 'El señor' or 'La señora'
        if len(tag) > 9:
            after_title = tag[9:colon_pos].lower().strip()
            if after_title:
                extractions.append(after_title)
                clean_tag = tag_lower.replace(':', '').replace(',', '').strip()
        extractions.append(clean_tag)
        
        # Calcular similitud máxima
        for extraction in extractions:
            if extraction:
                sim = SequenceMatcher(None, apellidos, extraction).ratio()
                if sim > best_sim:
                    best_sim = sim
                    best_match_name = candidate.get('apellidos')
                    best_grupo = candidate.get('formacion')
                    best_gender = candidate.get('genero')
                    best_diputado = candidate
    
    return {
        'name': best_match_name,
        'group': best_grupo,
        'gender': best_gender,
        'similarity': best_sim,
        'diputado': best_diputado
    }


In [ ]:
best_match('El señor FUGA IBARNE', diputados)

In [ ]:
def process_single_session(filepath: Path, deputies: List[Dict], institutional_roles: List[Dict],
                           threshold: float = 0.7) -> List[Tuple[str, str, str, float, str]]:
    results = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except Exception as e:
        print(f'Error reading {filepath}: {e}')
        return results
    
    if not lines:
        return results
    
    cache = []  # Session cache: deputies already found in this session (efficiency)
    
    # Process pairs line-by-line (name in even lines, intervention in odd lines)
    for i in range(0, len(lines) - 1, 2):
        tag = lines[i].strip()
        text = lines[i + 1].strip() if i + 1 < len(lines) else ''
        
        if not tag:
            continue
        
        # Verify institutional roles 
        role_match = best_match(tag, institutional_roles)
        if role_match['similarity'] > threshold:
            results.append((role_match['name'], role_match['group'], role_match['gender'],
                           role_match['similarity'], text))
            continue
        
        # Verify local cache (efficiency) 
        if cache:
            cache_match = best_match(tag, cache)
            if cache_match['similarity'] > threshold:
                results.append((cache_match['name'], cache_match['group'], cache_match['gender'],
                               cache_match['similarity'], text))
                continue
        
        # Search over full deputies list
        deputy_match = best_match(tag, deputies)
        if deputy_match['similarity'] > threshold:
            results.append((deputy_match['name'], deputy_match['group'], deputy_match['gender'],
                           deputy_match['similarity'], text))
            
            if deputy_match['diputado']:
                cache.append(deputy_match['diputado'])
        else:
            # Not found match
            results.append((None, None, None, deputy_match['similarity'], text))
    
    return results

In [ ]:
# Iteration over the whole dataset 

import os
from tqdm import tqdm

# Define the least similarity index to consider a match valid. 
SIMILARITY_THRESHOLD = 0.7

legislature_col = []
session_num_col = []
speaker_col = []
group_col = []
gender_col = []
similarity_col = []
text_col = []

if not INTERVENTION_TEXT_DIR.exists():
    print(f'Warning: {INTERVENTION_TEXT_DIR} does not exist. Creating empty dataframe.')
    interventions_df = pd.DataFrame()
else:
    for filepath in sorted(INTERVENTION_TEXT_DIR.glob('PL_*.txt')):
        try:
            # Extraer legislatura y número de sesión del nombre
            stem = filepath.stem  # e.g., 'PL_1_24'
            parts = stem.split('_')
            if len(parts) < 3:
                continue
            legislatura = int(parts[1])
            numero = int(parts[2])
            
            # Procesar este archivo
            interventions = process_single_session(filepath, diputados, cargos, SIMILARITY_THRESHOLD)
            
            for speaker, group, gender, sim, text in interventions:
                legislature_col.append(legislatura)
                session_num_col.append(numero)
                speaker_col.append(speaker)
                group_col.append(group)
                gender_col.append(gender)
                similarity_col.append(sim)
                text_col.append(text)
        except Exception as e:
            print(f'Error processing {filepath.name}: {e}')
            continue
    
    # Crear dataframe
    interventions_df = pd.DataFrame({
        'legislature': legislature_col,
        'session_number': session_num_col,
        'speaker': speaker_col,
        'group': group_col,
        'gender': gender_col,
        'text': text_col
    })
    
    print(f'Processed {len(interventions_df)} interventions')
    print(interventions_df.head())

### Data organization

Now that we have every match, we need to:
- get the date of each intervention by using the CSV obtained in notebook 01.
- normalize gender format

In [ ]:
# Get dates for each session and assign to interventions
dates_dict = {}
for idx, row in dates_df.iterrows():
    leg = row.iloc[0]  # Primera columna es legislatura
    sess = row.iloc[1]  # Segunda columna es número de sesión
    date_str = str(row.iloc[2]) if len(row) > 2 else None
    
    if date_str and len(date_str) == 8:  # Formato YYYYMMDD
        try:
            date = pd.Timestamp(year=int(date_str[:4]), month=int(date_str[4:6]), day=int(date_str[6:]))
            dates_dict[(leg, sess)] = date
        except Exception as e:
            print(f'Error parsing date {date_str}: {e}')
            continue

# Asignar fechas al dataframe
dates_list = []
for idx, row in interventions_df.iterrows():
    key = (row['legislature'], row['session_number'])
    dates_list.append(dates_dict.get(key, pd.NaT))

interventions_df['date'] = dates_list
print('Date column added')
print(interventions_df[['legislature', 'session_number', 'date']].head())

In [ ]:
# Normalizar columna de género
def normalize_gender(val):
    """Convierte códigos numéricos de género a strings legibles."""
    if val == 1:
        return 'Male'
    elif val == 2:
        return 'Female'
    else:
        return 'Unknown'

interventions_df['gender'] = interventions_df['gender'].apply(normalize_gender)

# Extraer año de la fecha
interventions_df['year'] = interventions_df['date'].dt.year

# Guardar resultado intermedio
unfiltered_output = OUTPUT_DIR / 'interventions_unfiltered.csv'
interventions_df.to_csv(unfiltered_output, index=False, encoding='utf-8')

### Data filtering and metadata normalizing

For the following research goals, we will simplify the politcal compass into 4 groups. (see methodology)

We will dismiss interventions which don't fit into this compass or give any other issues.

We'll drop the metadata we won't use (day/month, name of the speaker)

In [8]:
import pandas as pd
interventions_df = pd.read_csv(OUTPUT_DIR / 'interventions_unfiltered.csv')
interventions_filtered = pd.read_csv(OUTPUT_DIR / 'interventions_filtered.csv')

In [7]:
interventions_df.head()

,text,session_number,legislature,group,speaker,gender,date,year
0,De acuerdo con los datos que obran en la Secre...,1,1,NaN,NaN,NaN,1979-03-23,1979
1,"Señoras y señores Diputados, se abre la sesión...",1,1,CP,Fraile Poujade,Male,1979-03-23,1979
2,sEENERAL DEL CONGRESO DE LOS DIPUTADOS (Rubio ...,1,1,NEUTRAL,Secretario,NaN,1979-03-23,1979
3,El artículo que hace referencia a este acto es...,1,1,NaN,NaN,NaN,1979-03-23,1979
4,Según la relación que consta en la Secretaría ...,1,1,CP,Fraile Poujade,Male,1979-03-23,1979


In [9]:
interventions_filtered.head()

,text,group,gender,year,mistral_large,nfrases,acusacion
0,"Señoras y señores Diputados, se abre la sesión...",PP,Male,1979,0,3,0.0
1,Según la relación que consta en la Secretaría ...,PP,Male,1979,0,9,0.0
2,"Siguiendo el orden del día, ruego al señor Esp...",PSOE,Male,1979,0,4,0.0
3,Señoras y señores Diputados. en nombre de la M...,PSOE,Male,1979,0,18,0.0
4,"Al reanudarse la sesión, procedería dar lectur...",PSOE,Male,1979,0,7,0.0


In [ ]:
# Definir mapeo de grupos políticos
# Los grupos originales en los datos tienen formatos heterogéneos; unificamos en 4 categorías
PARTY_GROUPS = {
    'PP': {
        'PN-PP', 'PP-EU', 'PP-PAR', 'AP', 'PP-CG', 'CP', 'PP', 'UPM', 'PP-FORO', 'UPN'
    },
    'PSOE': {
        'PsdeG-PSOE', 'PSOE-A', 'PSE-EE', 'PSC(PSC-PSOE)', 'PSOEdeAndalucía',
        'PSC-PSOE', 'PSOE-progresistas', 'PSG-PSOE', 'PSE-EE-PSOE',
        'PSE-EE (PSOE)', 'Pse-EE(PSOE)', 'PSE-PSOE', 'PSOE-NCa', 'PSN-PSOE',
        'PSIB-PSOE', 'PSA', 'PSdG-PSOE', 'PSdeG-PSOE-progresistas', 'PSOE'
    },
    'VOX': {'Vox', 'VOX'},
    'IU-PODEMOS': {
        'IU-CA', 'IU-EUPV', 'PODEMOS', 'PCE', 'IU-CM', 'IU-EB', 'IU', 'UP',
        'SUMAR', 'IU-LV-CA', 'MÁS PAÍS-EQUO', 'IU-IX', 'IU-LV', 'IZQ-PLU', 'IU-A'
    }
}

def classify_party(subgroup: str) -> str:
    """Clasifica un subgrupo en una de las 4 categorías principales."""
    if pd.isna(subgroup):
        return None
    subgroup = str(subgroup).strip()
    for party, subgroups in PARTY_GROUPS.items():
        if subgroup in subgroups:
            return party
    return subgroup  # Retornar el original si no coincide

# Aplicar clasificación
interventions_df['party'] = interventions_df['group'].apply(classify_party)

In [ ]:
# Filtering: VOX didn't enter until December 2018
# Interventions labeled as VOX before 2018 come from deputies who changed party and are discarded
before_vox_mask = (interventions_df['party'] == 'VOX') & (interventions_df['year'] < 2018)
interventions_df.loc[before_vox_mask, 'party'] = None

# Filtering: Keep only interventions with identified speaker and party
filtered_df = interventions_df[
    (interventions_df['speaker'].notna()) &
    (interventions_df['party'].notna())
].copy()

In [ ]:
# Drop columns that are no longer needed

filtered_df = filtered_df.drop(columns=['group','date','speaker', 'session_number', 'legislature'])
filtered_df = filtered_df.rename(columns={'party': 'group'})

In [ ]:
# Save
filtered_output = OUTPUT_DIR / 'interventions_filtered.csv'
filtered_df.to_csv(filtered_output, index=False, encoding='utf-8')

## Notas de reproducibilidad

- **VERSIONS**: Registra las versiones de Python, pandas y spacy usadas en este análisis
- **PROCESS_DATE**: Timestamp UTC del procesamiento
- **SIMILARITY_THRESHOLD**: Parámetro que controla cuán estricto es el matching de nombres. Aumentar para mayor precisión, reducir para mayor cobertura.
- **Outputs**: Se guardan tres archivos CSV en `data/processed/`:
  - `interventions_raw.csv`: Datos brutos post-matching + fechas
  - `interventions_filtered.csv`: Filtrados, grupos consolidados, sin intervenciones pre-2018 de VOX
  - `interventions_final.csv`: Con análisis de language polarizado

Para reproducibilidad científica, guardar junto con los outputs los valores de `VERSIONS` y `PROCESS_DATE`.